In [10]:
# Import the packages needed inside the function.
import wrds
import pandas as pd


def build_financial_ratio_report(username, company_dict, start_year):
    """
    Download financial ratio data from WRDS and return four pivot tables.

    Parameters:
        username (str): WRDS username
        company_dict (dict): mapping from stock code to company name
                        
        start_year (int): keep data from this year onward

    Returns:
        dict: a dictionary containing the cleaned data and four markdown tables
    """

    # --- Connect to WRDS ---
    # A new connection is created inside the function so it is self-contained.
    conn = wrds.Connection(wrds_username=username)

    # --- Build the stock list for the SQL IN clause ---
    # .keys() returns all the dictionary keys (the stock codes).
    # tuple() converts the list of keys into a tuple.
    # SQL's IN clause expects this format: WHERE stkcd IN 
    stock_list = tuple(company_dict.keys())

    # --- Build the SQL query ---
    # This query calculates four financial ratios directly inside SQL using AS.
    #
    # AS keyword in SQL (renaming computed columns):
    # When a column is the result of a calculation 
    # it has no name by default. AS gives it a name so we can refer to it later.
    #
    # The five ratios:
    #   roa          = Net Profit / Total Assets
    #   roe          = Net Profit / Total Equity  
    #   profitmargin = Net Profit / Total Revenue 
    #   turnover     = Total Revenue / Total Assets 
    #   leverage     = Total Assets / Total Equity  
    sql = f"""
    SELECT
        stkcd,
        accper,
        b002000000 / a001000000 AS roa,
        b002000000 / a003000000 AS roe,
        b002000000 / b001101000 AS profitmargin,
        b001101000 / a001000000 AS turnover,
        a001000000 / a003000000 AS leverage
    FROM csmar.wrds_csmar_financial_master
    WHERE stkcd IN {stock_list}
      AND typrep = 'A'
    """

    # --- Run the query and convert accper to datetime ---
    data = conn.raw_sql(sql, date_cols=["accper"])

    # Always close the connection after downloading data.
    conn.close()

    # --- Filter to December (year-end) observations only ---
    # .dt.month == 12 keeps only rows where accper falls in December.
    # .copy() avoids a pandas warning when we modify the filtered DataFrame.
    df = data[data["accper"].dt.month == 12].copy()

    # Extract the year from accper and store it as a new 'year' column.
    df["year"] = df["accper"].dt.year

    # Keep only data from start_year onward.
    df = df[df["year"] >= start_year].copy()

    # Drop accper now that we have the 'year' column — it is no longer needed.
    df = df.drop(columns=["accper"])

    # --- Map stock codes to readable company names ---
    # .map(company_dict) looks up each value in the 'stkcd' column in the dictionary
    # and replaces it with the corresponding company name.
    df["Company"] = df["stkcd"].map(company_dict)

    # --- Create pivot tables ---
    #
    # pivot() parameters:
    #   index   = the column whose values become the row labels (here: Company)
    #   columns = the column whose values become the new column headers (here: year)
    #   values  = the column containing the numbers to fill the table (here: roe)
    #
    # After pivot(), we apply formatting:
    #   .mul(100)  → multiply by 100 to convert decimal to percentage (0.22 → 22)
    #   .round(1)  → round to 1 decimal place (22.140... → 22.1)
    #   .astype(str) + '%' → convert to string and add a '%' sign (22.1 → '22.1%')
    #
    # .rename_axis(columns=None) removes the label 'year' above the column headers.
    # .rename_axis('ROE', axis='index') renames the row index label to 'ROE'.

    # ROA pivot table
    roa_pivot = (
        df.pivot(index="Company", columns="year", values="roa")
        .mul(100)
        .round(1)
        .astype(str)
        + "%"
    )
    roa_pivot = roa_pivot.rename_axis(columns=None)
    roa_pivot = roa_pivot.rename_axis("ROA", axis="index")

    # ROE pivot table
    roe_pivot = (
        df.pivot(index="Company", columns="year", values="roe")
        .mul(100)
        .round(1)
        .astype(str)
        + "%"
    )
    roe_pivot = roe_pivot.rename_axis(columns=None)
    roe_pivot = roe_pivot.rename_axis("ROE", axis="index")

    # Profit Margin pivot table (same structure as ROE)
    pm_pivot = (
        df.pivot(index="Company", columns="year", values="profitmargin")
        .mul(100)
        .round(1)
        .astype(str)
        + "%"
    )
    pm_pivot = pm_pivot.rename_axis(columns=None)
    pm_pivot = pm_pivot.rename_axis("Profit Margin", axis="index")

    # Asset Turnover pivot table (same structure)
    to_pivot = (
        df.pivot(index="Company", columns="year", values="turnover")
        .mul(100)
        .round(1)
        .astype(str)
        + "%"
    )
    to_pivot = to_pivot.rename_axis(columns=None)
    to_pivot = to_pivot.rename_axis("Turnover", axis="index")

    # Leverage pivot table
    # Leverage is a ratio (e.g. 2.90), not a percentage, so we skip .mul(100) and '%'.
    lev_pivot = (
        df.pivot(index="Company", columns="year", values="leverage")
        .round(2)
    )
    lev_pivot = lev_pivot.rename_axis(columns=None)
    lev_pivot = lev_pivot.rename_axis("Leverage", axis="index")

    # --- Return all outputs packed into one dictionary ---
    # Returning a dictionary lets the caller access any specific piece they need
    # using result['roe_pivot'], result['pm_markdown'], etc.
    return {
        "raw_data":     df,  
        "roa_pivot":    roa_pivot,  
        "roe_pivot":    roe_pivot,     
        "pm_pivot":     pm_pivot,      
        "to_pivot":     to_pivot,      
        "lev_pivot":    lev_pivot, 
        "roa_markdown":  roa_pivot.to_markdown(),
        "roe_markdown":  roe_pivot.to_markdown(),
        "pm_markdown":  pm_pivot.to_markdown(),
        "to_markdown":  to_pivot.to_markdown(),
        "lev_markdown": lev_pivot.to_markdown(),
    }


In [21]:
# --- Define the inputs we will pass to the function ---

# Replace with your own WRDS username.
username = 'yuxunmei'#"your_wrds_username"

# company_dict maps each stock code (key) to a readable company name (value).
# These are three major Chinese home appliance manufacturers.
# You can add more companies or change the codes/names as needed.
company_dict = {
    '600519': 'Moutai',   
    '000858': 'Wuliangye',   
}

# Only keep financial data from this year onward (inclusive).
start_year = 2022

In [22]:
# --- Call the function with the parameters defined above ---
# The function connects to WRDS, downloads data, filters it,
# computes ratios, builds pivot tables, and returns everything in one dictionary.
# The returned dictionary is stored in 'result' for inspection below.

result = build_financial_ratio_report(
    username=username,
    company_dict=company_dict,
    start_year=start_year
)

Loading library list...
Done


In [14]:
# Confirm that 'result' is a Python dictionary.
# type() returns the data type of any Python object.
type(result)

dict

In [15]:
# List all the keys (output names) available in the result dictionary.
# This is a list comprehension: it creates a list by iterating over the keys.
# It is equivalent to: list(result.keys())
[key for key in result]

['raw_data',
 'roa_pivot',
 'roe_pivot',
 'pm_pivot',
 'to_pivot',
 'lev_pivot',
 'roa_markdown',
 'roe_markdown',
 'pm_markdown',
 'to_markdown',
 'lev_markdown']

In [23]:
# Access the 'raw_data' item from the result dictionary.
# result['raw_data'] is the cleaned long-format DataFrame
# with columns: stkcd, roe, profitmargin, turnover, leverage, year, Company.
result["raw_data"]

,stkcd,roa,roe,profitmargin,turnover,leverage,year,Company
114,000858,0.183156,0.239712,0.378142,0.484358,1.308787,2022,Wuliangye
119,000858,0.190535,0.238163,0.378528,0.503358,1.249972,2023,Wuliangye
124,000858,0.176324,0.243363,0.372228,0.473701,1.380201,2024,Wuliangye
232,600519,0.257013,0.318958,0.526795,0.487881,1.241018,2022,Moutai
237,600519,0.284274,0.34661,0.52488,0.541598,1.219279,2023,Moutai
242,600519,0.298834,0.369135,0.522734,0.571675,1.23525,2024,Moutai


In [24]:
#Download the raw data
result["raw_data"].to_csv("financial_ratios.csv", index=False)
print("The data has been saved as: financial_ratios.csv")

The data has been saved as: financial_ratios.csv


In [25]:
# Access the 'roa_markdown' item from the result dictionary.
print(result["roa_markdown"])

| ROA       | 2022   | 2023   | 2024   |
|:----------|:-------|:-------|:-------|
| Moutai    | 25.7%  | 28.4%  | 29.9%  |
| Wuliangye | 18.3%  | 19.1%  | 17.6%  |


In [26]:
print(result["roe_markdown"])

| ROE       | 2022   | 2023   | 2024   |
|:----------|:-------|:-------|:-------|
| Moutai    | 31.9%  | 34.7%  | 36.9%  |
| Wuliangye | 24.0%  | 23.8%  | 24.3%  |


In [27]:
print(result["pm_markdown"])

| Profit Margin   | 2022   | 2023   | 2024   |
|:----------------|:-------|:-------|:-------|
| Moutai          | 52.7%  | 52.5%  | 52.3%  |
| Wuliangye       | 37.8%  | 37.9%  | 37.2%  |


In [28]:
print(result["to_markdown"])

| Turnover   | 2022   | 2023   | 2024   |
|:-----------|:-------|:-------|:-------|
| Moutai     | 48.8%  | 54.2%  | 57.2%  |
| Wuliangye  | 48.4%  | 50.3%  | 47.4%  |


In [29]:
print(result["lev_markdown"])

| Leverage   |   2022 |   2023 |   2024 |
|:-----------|-------:|-------:|-------:|
| Moutai     |   1.24 |   1.22 |   1.24 |
| Wuliangye  |   1.31 |   1.25 |   1.38 |


In [ ]:
# Close the WRDS connection.
db.close()